<a href="https://colab.research.google.com/github/umachandola249/IIT_BS_Machine_Learning/blob/main/agriculture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

kaggriculture_path = kagglehub.competition_download('kaggriculture')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Conway's Go-Farming: Advanced Automation Agent
WELCOME to documentation and codebase for our competation submission.This notebook implements an intelligent state-machine automation agent designed to manage resources, evalution fluid market dynamics, and orchestrate a multi-worker workforce efficiently across the farm.

# Table of Contents
* [1. Global Configurations & Game Parameters](#1.-Global-Configurations-&-Game-Parameters)
* [2. Action Prioritization Matrix](#2.-Action-Prioritization-Matrix)
* [3. Navigation & Coordinate Pathfinding](#3.-Navigation-&-Coordinate-Pathfinding)
* [4. Dynamic Market Value Ratios](#4.-Dynamic-Market-Value-Ratios)
* [5. Crop Strategy & Seed Allocation](#5.-Crop-Strategy-&-Seed-Allocation)
* [6. Real-Time Tile Evaluation Engine](#6.-Real-Time-Tile-Evaluation-Engine)
* [7. Worker Task Translation](#7.-Worker-Task-Translation)
* [8. Multi-Unit Target Scanning & Grid Resolution](#8.-Multi-Unit-Target-Scanning-&-Grid-Resolution)
* [9. Core Worker Orchestration Layer](#9.-Core-Worker-Orchestration-Layer)
* [10. Main Agent Entry Point (`agent`)](#10.-Main-Agent-Entry-Point-(agent))

%%capture
!pip install kaggle_environments

In [ ]:
!pip install kaggle

In [ ]:
!pip install kaggle-environments

# 1. Global Configuration & Game Parameters
In this section, we define the baseline economic constatnts of the environment. Thia includes tracking available crops ('WHEAT' , 'CARROT','TOMATO','STRAWBERRY','MELON'), estimating seed purchase fees, establishing standard item base, and implement trading safeguards( such as maximum order chunks and safty price floors.)

In [ ]:
CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]

# Only WHEAT's cost is confirmed (from the official Quick Start example).
# The rest are estimates, roughly scaled to each crop's base price.
SEED_COSTS = {
    "WHEAT": 10,
    "CARROT": 15,
    "TOMATO": 25,
    "STRAWBERRY": 50,
    "MELON": 100,
}

# Confirmed base prices from the Price Function table in "How to Play".
BASE_PRICES = {
    "WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
    "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100,
}

SELL_CHUNK = 20                  # cap per SELL order so we don't crash our own price
MIN_SELL_PRICE_FRACTION = 0.5    # don't sell below 50% of a resource's base price
SEED_STOCK_BUFFER = 2            # keep at most this many spare seeds of the "best" crop

# 2. Action Prioritization Matrix

To prevent catastrophic losses (like losing livestock to starvation or letting ripe crops turn into weeds), we establish a strict PURPOSE_PRIORITY hierarchy. Lower numerical ranks ensure critical survival tasks are executed before expansionary behaviors like planting.

In [ ]:
# Lower number = handled first when several tiles need attention.
PURPOSE_PRIORITY = {
    "feed": 0,                # 2 days unfed -> animal is lost for good
    "harvest": 1,
    "harvest_animal": 1,
    "water": 2,                # 2 days unwatered -> plant becomes a weed
    "collect_fertilizer": 3,   # uncollected fertilizer does not accumulate
    "dig": 4,
    "care": 5,
    "plant": 6,
}

# 3. Navigation & Coordinate Pathfinding

Implements the _step_toward navigational function. This script compares the current grid location against the targeted coordinates, prioritizing horizontal adjustments before correcting vertical alignment to close distances step-by-step.

In [ ]:
def _step_toward(fx, fy, tx, ty):
    """One cardinal step toward (tx, ty). CONFIRMED action words: there is
    no separate MOVE verb -- the direction word IS the action."""
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None  # already there

# 4. Dynamic Market Value Ratios

Defines the economic valuation engine (_crop_value_ratio). This script continuously evaluates current fluid market prices against baseline values to pinpoint market inflation spikes, allowing the agent to capture peak profit margins.

In [ ]:
def _crop_value_ratio(crop, prices):
    """How good a deal is this crop selling for right now, relative to its
    own base price? A ratio above 1 means it is currently fetching more
    than usual."""
    base = BASE_PRICES.get(crop, 1) or 1
    return prices.get(crop, 0) / base

# 5. Crop Strategy & Seed Allocation

Handles seed selection logic for both planting (_crop_to_plant) and purchasing (_crop_to_buy_seed_for). It relies on a crop diversification strategy -- first ensuring all crop varieties are unlocked, then focusing cultivation on the most valuable assets.

In [ ]:
def _crop_to_plant(seeds, prices):
    """Diversification: among crops we ALREADY hold seed for, plant
    whichever one currently commands the best price. Returns None if we
    hold no seed at all."""
    held = [c for c in CROPS if seeds.get(c, 0) > 0]
    if not held:
        return None
    return max(held, key=lambda c: _crop_value_ratio(c, prices))


def _crop_to_buy_seed_for(seeds, prices):
    """Diversification: prefer buying seed for a crop we don't hold yet
    at all (spreads us across crop types); once every crop is held,
    just top up whichever is currently most valuable."""
    unheld = [c for c in CROPS if seeds.get(c, 0) == 0]
    pool = unheld if unheld else CROPS
    return max(pool, key=lambda c: _crop_value_ratio(c, prices))

# 6. Real-Time Tile Evaluation Engine

The core state-machine diagnostic tool (_tile_purpose). As units move or scan, this function inspects the specific dictionary payload of a tile to verify if it requires immediate harvesting, weeding, watering, livestock feeding, or new seed sowing.

In [ ]:
def _tile_purpose(tile, seeds, prices):
    """What would be worth doing if we were standing on this tile right
    now? Returns (purpose, crop_or_None) or None."""
    if tile is None:
        crop = _crop_to_plant(seeds, prices)
        return ("plant", crop) if crop else None
    if not isinstance(tile, dict):
        return None  # e.g. "LOCKED"

    kind = tile.get("kind")
    if kind == "PLANT":
        if tile.get("yield_units", 0) > 0:
            return ("harvest", None)
        if not tile.get("watered_today", False):
            return ("water", None)
        return None
    if kind == "WEED":
        return ("dig", None)
    if kind in ("COOP", "PASTURE") and tile.get("animal"):
        if not tile.get("fed_today", False):
            return ("feed", None)
        if tile.get("yield_units", 0) > 0:
            return ("harvest_animal", None)
        if tile.get("fertilizer_available", False):
            return ("collect_fertilizer", None)
        if not tile.get("cared_today", False):
            return ("care", None)
        return None
    return None

# 8. Multi-Unit Target Scanning & Grid Resolution

Implements _find_target, which loops systematically through the entire matrix grid using Manhattan Distance rules. It identifies the highest-priority task closest to the unit while strictly ignoring tiles already locked by other friendly workers.

# 7. Worker Task Translation

Converts internal abstract target mission strings (like "harvest" or "water") into standardized game-engine list command arrays (like ["HARVEST"] or ["PLANT", crop]) that the environment can parse successfully.

In [ ]:
def _action_for_purpose(purpose, crop):
    if purpose == "plant":
        return ["PLANT", crop]
    return {
        "harvest": ["HARVEST"],
        "harvest_animal": ["HARVEST"],
        "water": ["WATER"],
        "dig": ["DIG"],
        "feed": ["FEED"],
        "collect_fertilizer": ["COLLECT_FERTILIZER"],
        "care": ["CARE"],
    }.get(purpose, ["PASS"])

In [ ]:
def _find_target(farm, board_size, seeds, prices, claimed, from_xy):
    """Find the highest-priority, then nearest, tile worth walking to."""
    fx, fy = from_xy
    candidates = []
    for y in range(board_size):
        for x in range(board_size):
            if (x, y) in claimed:
                continue
            result = _tile_purpose(farm["tiles"][y][x], seeds, prices)
            if result:
                purpose, crop = result
                candidates.append((x, y, purpose, crop))
    if not candidates:
        return None
    candidates.sort(
        key=lambda c: (PURPOSE_PRIORITY[c[2]], abs(c[0] - fx) + abs(c[1] - fy))
    )
    return candidates[0]

# 9. Core Worker Orchestration Layer

Defines _move_unit, managing the turn execution flow for an individual entity. It acquires targets, registers locked coordinates to avoid duplication of work, and issues the ultimate step or action command.

In [ ]:
def _move_unit(farm, board_size, seeds, prices, claimed, position):
    """Decide one unit's (farmer's or a hand's) action for this turn."""
    x, y = position
    here = _tile_purpose(farm["tiles"][y][x], seeds, prices)
    if here:
        purpose, crop = here
        claimed.add((x, y))
        return _action_for_purpose(purpose, crop)

    target = _find_target(farm, board_size, seeds, prices, claimed, (x, y))
    if target is None:
        return ["PASS"]
    tx, ty, purpose, crop = target
    claimed.add((tx, ty))
    step = _step_toward(x, y, tx, ty)
    return [step] if step else ["PASS"]

# 10. Main Agent Entry Point (agent)

The global runtime handler function. It unpacks the environment's obs frame, runs the automated marketplace trading cycle, dynamically buys utilities (like fertilizer) or hires more hands when milestones are met, triggers unit movements, and packages the final transaction dictionary to return to the simulation framework.

In [ ]:
def agent(obs):
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]

    money = me.get("money", 0)
    board_size = len(me["tiles"])
    fx, fy = me["farmer"]
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    prices = obs.get("market", {}).get("prices", {})

    market_orders = []

    # --- Sell shed contents: chunked, and only at a decent price ----------
    for item, qty in shed.items():
        if item == "FERTILIZER" or not qty:
            continue  # keep fertilizer -- it can't be bought back easily
        base = BASE_PRICES.get(item, 0)
        current_price = prices.get(item, 0)
        if base and current_price < base * MIN_SELL_PRICE_FRACTION:
            continue  # market's paying too little right now; hold and wait
        sell_qty = min(qty, SELL_CHUNK)
        if sell_qty > 0:
            market_orders.append(["SELL", item, sell_qty])

    # --- Keep a small fertilizer buffer topped up --------------------------
    fert_price = prices.get("FERTILIZER", 9999)
    if money > 500 and fert_price <= 150 and shed.get("FERTILIZER", 0) < 10:
        market_orders.append(["BUY_PRODUCT", "FERTILIZER", 2])

    # --- Hire extra hands once we can comfortably afford it -----------------
    hires_today = me.get("hires_today", 0)
    if money > 3000 and hires_today < 1:
        market_orders.append(["HIRE"])
    elif money > 8000 and hires_today < 2:
        market_orders.append(["HIRE"])

    # --- Diversified seed buying ---------------------------------------------
    buy_crop = _crop_to_buy_seed_for(seeds, prices)
    if buy_crop and seeds.get(buy_crop, 0) < SEED_STOCK_BUFFER:
        cost = SEED_COSTS.get(buy_crop, 9999)
        if money >= cost:
            market_orders.append(["BUY_SEED", buy_crop, 1])

    # --- Move/act: farmer first, then each hand, avoiding duplicate work ---
    claimed = set()
    farmer_action = _move_unit(me, board_size, seeds, prices, claimed, (fx, fy))

    hands_actions = [
        _move_unit(me, board_size, seeds, prices, claimed, (hx, hy))
        for hx, hy in me.get("hands", [])
    ]

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market_orders,
    }

## 11. Worker Specialization Layer

To reduce idle turns and overlapping tasks, each unit is assigned a role:
- Farmer → planting & harvesting
- Crop Worker → watering & weeding
- Livestock Worker → feeding & caring for animals
- Expansion Worker → digging new plots

This specialization ensures that every hand contributes efficiently, lowering wasted actions and improving overall farm productivity.

In [ ]:
def assign_role(index):
    roles = ["farmer","crop_worker","livestock_worker","expansion_worker"]
    return roles[index % len(roles)]

def specialized_move_unit(farm, board_size, seeds, prices, claimed, position, role):
    x, y = position
    here = _tile_purpose(farm["tiles"][y][x], seeds, prices)
    if role == "livestock_worker" and here and here[0] in ("feed","care","collect_fertilizer"):
        claimed.add((x,y)); return _action_for_purpose(*here)
    if role == "crop_worker" and here and here[0] in ("plant","water","harvest"):
        claimed.add((x,y)); return _action_for_purpose(*here)
    target = _find_target(farm, board_size, seeds, prices, claimed, (x,y))
    if not target: return ["PASS"]
    tx, ty, purpose, crop = target; claimed.add((tx,ty))
    step = _step_toward(x,y,tx,ty)
    return [step] if step else ["PASS"]

## 12. Metrics & Performance Dashboard

To evaluate the agent’s effectiveness, we track:
- Profit trend → total shed value × market prices
- Idle turns → number of ["PASS"] actions
- Crop diversity → number of distinct crops grown

These metrics are plotted after each run to visualize improvements and guide parameter tuning.

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

metrics = {"profit_history":[],"idle_turns":0,"crop_diversity":set()}

def log_metrics(action_dict, shed, prices):
    if action_dict.get("farmer") == ["PASS"]:
        metrics["idle_turns"] += 1
    profit = sum(qty * prices.get(item,0) for item, qty in shed.items())
    metrics["profit_history"].append(profit)
    for crop in shed.keys():
        if crop in CROPS:
            metrics["crop_diversity"].add(crop)

def plot_metrics():
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.plot(metrics["profit_history"])
    plt.title("Profit Trend")
    plt.subplot(1,3,2)
    plt.bar(["Idle Turns"],[metrics["idle_turns"]])
    plt.title("Idle Turns")
    plt.subplot(1,3,3)
    plt.bar(["Crop Diversity"],[len(metrics["crop_diversity"])])
    plt.title("Crop Diversity")
    plt.show()

## 13. Predictive Pricing & Fertilizer Buffer

Two key upgrades improve stability:

- **Predictive Pricing**: Smooths out noisy one‑turn spikes by blending current price with a rolling average. This prevents bad sell decisions.
- **Fertilizer Buffer**: Keeps a small reserve of fertilizer (5–10 units) by buying when cheap. This boosts crop and livestock yields consistently.

In [ ]:
# --- Predictive Pricing ---
PRICE_HISTORY_LEN = 5
PRICE_ITEMS = list(BASE_PRICES.keys())
_price_history = {item: [] for item in PRICE_ITEMS}

def _update_price_history(prices):
    for item in PRICE_ITEMS:
        history = _price_history[item]
        history.append(prices.get(item, BASE_PRICES.get(item, 0)))
        if len(history) > PRICE_HISTORY_LEN:
            history.pop(0)

def _rolling_average(item):
    history = _price_history[item]
    if not history:
        return BASE_PRICES.get(item, 0)
    return sum(history) / len(history)

def _blended_price(item, prices):
    current = prices.get(item, BASE_PRICES.get(item, 0))
    return 0.5 * current + 0.5 * _rolling_average(item)

# --- Fertilizer Buffer ---
def fertilizer_logic(shed, prices, money, market_orders):
    fert_price = prices.get("FERTILIZER", 9999)
    if money > 500 and fert_price <= 150 and shed.get("FERTILIZER", 0) < 10:
        market_orders.append(["BUY_PRODUCT", "FERTILIZER", 2])
    return market_orders

NameError: name 'BASE_PRICES' is not defined

## 14. Hiring & Seed Diversification

These rules expand workforce and ensure crop diversity:

- **Hiring Thresholds**: Hire extra hands once money > 3000 (first hand) and > 8000 (second hand).
- **Seed Diversification**: Prefer buying seeds for crops not yet held, then top up whichever crop is most valuable. This keeps the farm resilient against market swings.

In [ ]:
def hiring_logic(money, hires_today, market_orders):
    if money > 3000 and hires_today < 1:
        market_orders.append(["HIRE"])
    elif money > 8000 and hires_today < 2:
        market_orders.append(["HIRE"])
    return market_orders

def _crop_value_ratio(crop, prices):
    base = BASE_PRICES.get(crop, 1) or 1
    return _blended_price(crop, prices) / base

def _crop_to_buy_seed_for(seeds, prices):
    unheld = [c for c in CROPS if seeds.get(c, 0) == 0]
    pool = unheld if unheld else CROPS
    return max(pool, key=lambda c: _crop_value_ratio(c, prices))

# Assembling main.py

Every function above was defined live in this kernel, one concept at a time, so you could follow the design. The cell below writes the exact same code out to main.py as a single file -- that's the actual file Kaggle will run when this agent is submitted.

In [ ]:
# This cell writes the complete, self-contained main.py. Copy this whole block into one cell exactly as shown (the %%writefile main.py line must be the first line of the cell):

# This cell writes the complete, self-contained main.py. Copy this whole block into one cell exactly as shown (the %%writefile main.py line must be the first line of the cell):

In [ ]:
%%writefile main.py
"""
Kaggriculture agent (v3 -- crop diversification)
===================================================

Builds on v2 (confirmed locally: 1679 vs 50 against the random baseline,
zero rejected actions over a full 720-turn game). This version's one
change: the agent now grows more than one crop instead of locking onto
whichever single crop it happens to hold seed for.

CONFIRMED mechanics in play here (see v2 for the full derivation):
- Movement = a direction word as the action: ["NORTH"/"SOUTH"/"EAST"/"WEST"].
- Weed tiles: ["DIG"]. Plants: PLANT <crop>, WATER, HARVEST.
- Animals: FEED, HARVEST, COLLECT_FERTILIZER, CARE.
- Market orders: BUY_SEED <crop> <n>, BUY_PRODUCT <WHEAT|FERTILIZER> <n>,
  SELL <item> <n>, HIRE.

WHAT'S NEW IN v3:
- _crop_value_ratio(): current price / base price for a crop -- a simple
  "is this crop unusually valuable right now?" signal.
- _crop_to_buy_seed_for(): prefers buying seed for a crop we don't yet
  hold at all, breaking ties toward whichever has the best price ratio.
  Once we hold every crop, it just tops up whichever is most valuable.
- _crop_to_plant(): when standing on an empty tile, plants whichever
  crop we're CURRENTLY holding seed for that has the best price ratio
  right now -- so different tiles can end up growing different crops.

STILL UNVERIFIED / NOT IMPLEMENTED (unchanged from v2, search "UNVERIFIED"):
- No confirmed action to fertilize a *plant* tile.
- Land expansion (buying the other 3 quadrants) isn't implemented.
- Seed costs beyond WHEAT ($10, confirmed) are still guesses.

Test locally:
    python -c "
from kaggle_environments import make
env = make('kaggriculture', debug=True)
env.run(['main.py', 'random'])
print([(i, s.reward) for i, s in enumerate(env.steps[-1])])
"
"""

# NOTE: identical to the CROPS/etc. definitions in Section 1 above --
# this file just needs to be self-contained on its own.
CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]

# Only WHEAT's cost is confirmed (from the official Quick Start example).
# The rest are unverified guesses, roughly scaled to each crop's base price.
SEED_COSTS = {
    "WHEAT": 10,
    "CARROT": 15,
    "TOMATO": 25,
    "STRAWBERRY": 50,
    "MELON": 100,
}

# Confirmed base prices from the Price Function table in "How to Play".
BASE_PRICES = {
    "WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
    "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100,
}

SELL_CHUNK = 20                  # cap per SELL order so we don't crash our own price
MIN_SELL_PRICE_FRACTION = 0.5    # don't sell below 50% of a resource's base price
SEED_STOCK_BUFFER = 2            # keep at most this many spare seeds of the "best" crop

# Lower number = handled first when several tiles need attention.
PURPOSE_PRIORITY = {
    "feed": 0,               # 2 days unfed -> animal is lost for good
    "harvest": 1,
    "harvest_animal": 1,
    "water": 2,               # 2 days unwatered -> plant becomes a weed
    "collect_fertilizer": 3,  # uncollected fertilizer does not accumulate
    "dig": 4,
    "care": 5,
    "plant": 6,
}


def _step_toward(fx, fy, tx, ty):
    """One cardinal step toward (tx, ty). CONFIRMED action words."""
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None  # already there


def _crop_value_ratio(crop, prices):
    """How good a deal is this crop selling for right now, relative to its
    own base price? >1 means it's currently fetching more than usual."""
    base = BASE_PRICES.get(crop, 1) or 1
    return prices.get(crop, 0) / base


def _crop_to_plant(seeds, prices):
    """Diversification: among crops we ALREADY hold seed for, plant
    whichever one currently commands the best price. Returns None if we
    hold no seed at all."""
    held = [c for c in CROPS if seeds.get(c, 0) > 0]
    if not held:
        return None
    return max(held, key=lambda c: _crop_value_ratio(c, prices))


def _crop_to_buy_seed_for(seeds, prices):
    """Diversification: prefer buying seed for a crop we don't hold yet
    at all (spreads us across crop types); once every crop is held,
    just top up whichever is currently most valuable."""
    unheld = [c for c in CROPS if seeds.get(c, 0) == 0]
    pool = unheld if unheld else CROPS
    return max(pool, key=lambda c: _crop_value_ratio(c, prices))


def _tile_purpose(tile, seeds, prices):
    """What would be worth doing if we were standing on this tile right
    now? Returns (purpose, crop_or_None) or None."""
    if tile is None:
        crop = _crop_to_plant(seeds, prices)
        return ("plant", crop) if crop else None
    if not isinstance(tile, dict):
        return None  # e.g. "LOCKED"

    kind = tile.get("kind")
    if kind == "PLANT":
        if tile.get("yield_units", 0) > 0:
            return ("harvest", None)
        if not tile.get("watered_today", False):
            return ("water", None)
        return None
    if kind == "WEED":
        return ("dig", None)
    if kind in ("COOP", "PASTURE") and tile.get("animal"):
        if not tile.get("fed_today", False):
            return ("feed", None)
        if tile.get("yield_units", 0) > 0:
            return ("harvest_animal", None)
        if tile.get("fertilizer_available", False):
            return ("collect_fertilizer", None)
        if not tile.get("cared_today", False):
            return ("care", None)
        return None
    return None


def _action_for_purpose(purpose, crop):
    if purpose == "plant":
        return ["PLANT", crop]
    return {
        "harvest": ["HARVEST"],
        "harvest_animal": ["HARVEST"],
        "water": ["WATER"],
        "dig": ["DIG"],
        "feed": ["FEED"],
        "collect_fertilizer": ["COLLECT_FERTILIZER"],
        "care": ["CARE"],
    }.get(purpose, ["PASS"])


def _find_target(farm, board_size, seeds, prices, claimed, from_xy):
    """Find the highest-priority, then nearest, tile worth walking to."""
    fx, fy = from_xy
    candidates = []
    for y in range(board_size):
        for x in range(board_size):
            if (x, y) in claimed:
                continue
            result = _tile_purpose(farm["tiles"][y][x], seeds, prices)
            if result:
                purpose, crop = result
                candidates.append((x, y, purpose, crop))
    if not candidates:
        return None
    candidates.sort(
        key=lambda c: (PURPOSE_PRIORITY[c[2]], abs(c[0] - fx) + abs(c[1] - fy))
    )
    return candidates[0]


def _move_unit(farm, board_size, seeds, prices, claimed, position):
    """Decide one unit's (farmer's or a hand's) action for this turn."""
    x, y = position
    here = _tile_purpose(farm["tiles"][y][x], seeds, prices)
    if here:
        purpose, crop = here
        claimed.add((x, y))
        return _action_for_purpose(purpose, crop)

    target = _find_target(farm, board_size, seeds, prices, claimed, (x, y))
    if target is None:
        return ["PASS"]
    tx, ty, purpose, crop = target
    claimed.add((tx, ty))
    step = _step_toward(x, y, tx, ty)
    return [step] if step else ["PASS"]


def agent(obs):
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]

    money = me.get("money", 0)
    board_size = len(me["tiles"])
    fx, fy = me["farmer"]
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    prices = obs.get("market", {}).get("prices", {})

    market_orders = []

    # --- 1. Sell shed contents: chunked, and only at a decent price -------
    for item, qty in shed.items():
        if item == "FERTILIZER" or not qty:
            continue  # keep fertilizer -- it can't be bought back easily
        base = BASE_PRICES.get(item, 0)
        current_price = prices.get(item, 0)
        if base and current_price < base * MIN_SELL_PRICE_FRACTION:
            continue  # market's paying too little right now; hold and wait
        sell_qty = min(qty, SELL_CHUNK)
        if sell_qty > 0:
            market_orders.append(["SELL", item, sell_qty])

    # --- 2. Keep a small fertilizer buffer topped up -----------------------
    fert_price = prices.get("FERTILIZER", 9999)
    if money > 500 and fert_price <= 150 and shed.get("FERTILIZER", 0) < 10:
        market_orders.append(["BUY_PRODUCT", "FERTILIZER", 2])

    # --- 3. Hire extra hands once we can comfortably afford it -------------
    hires_today = me.get("hires_today", 0)
    if money > 3000 and hires_today < 1:
        market_orders.append(["HIRE"])
    elif money > 8000 and hires_today < 2:
        market_orders.append(["HIRE"])

    # --- 4. Diversified seed buying -----------------------------------------
    buy_crop = _crop_to_buy_seed_for(seeds, prices)
    if buy_crop and seeds.get(buy_crop, 0) < SEED_STOCK_BUFFER:
        cost = SEED_COSTS.get(buy_crop, 9999)
        if money >= cost:
            market_orders.append(["BUY_SEED", buy_crop, 1])

    # --- 5. Move/act: farmer first, then each hand, avoiding duplicate work
    claimed = set()
    farmer_action = _move_unit(me, board_size, seeds, prices, claimed, (fx, fy))

    hands_actions = [
        _move_unit(me, board_size, seeds, prices, claimed, (hx, hy))
        for hx, hy in me.get("hands", [])
    ]

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market_orders,
    }

# Test it locally

Run one full 720-turn game against the built-in random agent, and confirm the agent produced no errors along the way.

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", debug=True)
env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

errors = sum(1 for entry in env.logs for log in entry if log.get("stderr"))
print("turns with stderr output:", errors)

# Known Limitations
No confirmed action exists yet to apply fertilizer to a plant tile -- fertilizer is purchased but never used on crops.
Land expansion (buying the other 3 quadrants) is not implemented.
Seed costs beyond WHEAT ($10, confirmed) are estimates.

In [ ]:
!kaggle competitions submit kaggriculture -f main.py -m "v3 diversified crops"
!kaggle competitions submissions kaggriculture
!kaggle competitions episodes <SUBMISSION_ID>